In [2]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('essay-gap-aicc-round-2')

print("Path to competition files:", path)

100%|██████████| 163k/163k [00:00<00:00, 447kB/s]

Extracting files...
Path to competition files: C:\Users\raian\.cache\kagglehub\competitions\essay-gap-aicc-round-2


In [18]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForMultipleChoice, AutoTokenizer

In [5]:
train_df = pd.read_csv(path + '/essay-gap/train.csv')
test_df = pd.read_csv(path + '/essay-gap/test.csv')
train_df

,sampleID,before,after,opt_0,opt_1,opt_2,opt_3,label
0,0,The life cycle of a Christmas tree from the se...,One issue that farmers face is the destruction...,The remaining development of the tree greatly ...,The belief in the divinity of Jesus leads to t...,Essentially the recipe brings together what tr...,It is a matter of some debate as to which was ...,0
1,1,Slopes flatter than 25 degrees or steeper than...,The rule of thumb is: A slope that is flat eno...,In her 1850 book The First Christmas in New En...,"In Latin America and the Iberian Peninsula, th...","On steeper slopes, this can occur with as litt...",When the incidence of human triggered avalanch...,3
2,2,"Most workplaces conduct a ""Christmas Party"" so...","Likewise, schools, TAFE (vocational training),...",As many people take their holidays between Chr...,The frequency with which avalanches form in a ...,"In doing so, they employ on-the-ground physica...",The area in and around the basilica begins to ...,0
3,3,The Chronography of 354 illuminated manuscript...,By around 385 the feast for the birth of Jesus...,The eastern inland region where the country is...,In a sermon delivered in Antioch on December 2...,This remains one of the most extensive such ma...,"A cold front, the leading edge of a cooler mas...",1
4,4,English personifications of Christmas were fir...,His character was maintained during the late 1...,"In a sermon in 386, Gregory of Nyssa specifica...",The first evidence of decorated trees associat...,"Following the Restoration in 1660, Father Chri...","In 614, the Persian Sassanid Empire, supported...",2
...,...,...,...,...,...,...,...,...
315,315,"Snow accumulates from a series of snow events,...",At some automatic weather stations an ultrason...,People can become lost in their own front yard...,Both types of gauges melt the accumulated snow...,'happy Christmas').,Greek children get their presents from Saint B...,1
316,316,"""Jingle Bells"" was first recorded by banjoist ...",There is a version by the Hayden Quartet calle...,"In Tromsø, Norway, a city located at 69 degree...",The earliest surviving vocal recording was mad...,Among earlier authors who influenced Dickens w...,A number of ecumenical councils were convened ...,1
317,317,"Michael Lloyd, and with most of the casting be...",Animation production services for the film wer...,In 2009 Vatican officials scheduled the Midnig...,Barren-ground caribou are susceptible to the e...,Among the all-star cast of voices were America...,Starting in 1983 with Vacanze di Natale ('Chri...,2
318,318,Some species of mammals hibernate while gestat...,"During hibernation, they subsequently lose 15–...","However, instead of building snowmen, the peop...","Some species, such as Viscum capense, are adap...",The Neolithic Goseck Circle in Germany has two...,The fat accumulation enables them to provide a...,3


In [ ]:
device = 'cuda'
endpoint = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(endpoint)
model = AutoModelForMultipleChoice.from_pretrained(endpoint, num_labels=4).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
class EssayDataset(Dataset):
    def __init__(self, df, tokenizer, is_train, max_len=512):
        super().__init__()
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        row = self.df.iloc[index]
        first  = [row['before']] * 4
        second = [f"{row[f'opt_{i}']} {row['after']}" for i in range(4)]

        enc = self.tokenizer(first, second, max_length=self.max_len, truncation='only_first')
        out = {**enc}
        if self.is_train:
            out['label'] = int(row['label'])
        return out

tr_df, va_df = train_test_split(train_df, test_size=0.15, random_state=42, stratify=train_df['label'])
train_ds = EssayDataset(tr_df, tokenizer, True)
val_ds = EssayDataset(va_df, tokenizer, True)
test_ds = EssayDataset(test_df, tokenizer, False)

In [20]:
from transformers import TrainingArguments, Trainer, DataCollatorForMultipleChoice

def compute_metrics(outputs):
    logits, labels = outputs
    preds = np.argmax(logits, axis=-1)
    return accuracy_score(labels, preds)

args = TrainingArguments(
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    logging_strategy='epoch',
    eval_strategy='epoch',
)

trainer = Trainer(model, args, DataCollatorForMultipleChoice(tokenizer, padding=True, max_length=512), train_ds, val_ds)

trainer.train()

c:\Users\raian\source\repos\AI\.env\Lib\site-packages\transformers\tokenization_utils_base.py:2353: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,1.381701,1.273444
2,0.948510,0.559115
3,0.483307,0.426069


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\raian\source\repos\AI\.env\Lib\site-packages\transformers\tokenization_utils_base.py:2353: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


TrainOutput(global_step=102, training_loss=0.9378394145591586, metrics={'train_runtime': 577.5112, 'train_samples_per_second': 1.413, 'train_steps_per_second': 0.177, 'total_flos': 324149909766144.0, 'train_loss': 0.9378394145591586, 'epoch': 3.0})